<a href="https://www.kaggle.com/code/ghadaataoui/eeg-signal-processing?scriptVersionId=248676768" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pickle
import numpy as np
import os

def read_data(filename):
    with open(filename, 'rb') as f:
        x = pickle._Unpickler(f)
        x.encoding = 'latin1'
        data = x.load()
    return data

# List of participant file names
files = [f"{i:02}" for i in range(1, 33)]

labels = []
data = []

base_path = "/kaggle/input/deap-dataset/deap-dataset/data_preprocessed_python/"

for i in files:
    file_path = os.path.join(base_path, f"s{i}.dat")
    d = read_data(file_path)
    labels.append(d['labels'])
    data.append(d['data'])

Convertir les données au format tableau
Convert data to table format

In [ ]:
labels = np.array(labels)
data = np.array(data)
print(labels.shape)
print(data.shape)

In [ ]:
labels = labels.reshape(1280, 4)
data = data.reshape(1280, 40, 8064)
print(labels.shape)
print(data.shape)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/participant_ratings.xls'
data = pd.read_excel(file_path, index_col=0)

threshold = 4.5
data['Valence_Category'] = ['V+' if val > threshold else 'V-' for val in data['Valence']]
data['Arousal_Category'] = ['A+' if ar > threshold else 'A-' for ar in data['Arousal']]

data['Category'] = data['Arousal_Category'] + ', ' + data['Valence_Category']

# Plot
plt.figure(figsize=(10, 6))

# Scatter plot with black points
plt.scatter(data['Valence'], data['Arousal'], color='black', alpha=0.7)

plt.title('Valence vs Arousal')
plt.xlabel('Valence')
plt.ylabel('Arousal')

# Add thresholds
plt.axhline(threshold, color='black', linestyle='--', linewidth=1, alpha=0.7)
plt.axvline(threshold, color='black', linestyle='--', linewidth=1, alpha=0.7)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

## Fusion au niveau des décisions (Decision Fusion)

# **Feature Extraction**

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
from scipy.signal import welch
from tqdm import tqdm

def bandpower(data, sf, band):
    band = np.asarray(band)
    low, high = band
    nperseg = (2 / low) * sf
    freqs, psd = welch(data, sf, nperseg=nperseg)
    freq_res = freqs[1] - freqs[0]
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    bp = np.trapz(psd[idx_band], dx=freq_res)
    return bp

def get_band_power(data, sf, band):
    bands = {
        "alpha": (8, 12),
        "beta": (12, 30),
        "gamma": (30, 64)
    }
    return bandpower(data, sf, bands[band]) if band in bands else None

# Paths
base_path = '/kaggle/input/deap-dataset/deap-dataset/data_preprocessed_python/'
output_base = '/kaggle/working/'

EEG_ch_names = ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1',
                'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8',
                'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2']
fs = 128

for channel_name in EEG_ch_names:
    channel_dir = os.path.join(output_base, channel_name)
    os.makedirs(channel_dir, exist_ok=True)

    for participant in tqdm(range(1, 33), desc=f'Processing channel {channel_name}'):
        file_path = os.path.join(base_path, f's{participant:02d}.dat')

        try:
            with open(file_path, 'rb') as f:
                raw = pickle.load(f, encoding='latin1')
                key = 'data' if 'data' in raw else b'data' if b'data' in raw else None
                if key is None:
                    raise KeyError("Key 'data' not found in file")
                data = raw[key]
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
            continue

        participant_data = []
        for trial in range(40):
            eeg_signal = data[trial, EEG_ch_names.index(channel_name), :]

            alpha_power = get_band_power(eeg_signal, fs, "alpha")
            beta_power = get_band_power(eeg_signal, fs, "beta")
            gamma_power = get_band_power(eeg_signal, fs, "gamma")

            combined_features = {
                'trial': trial,
                'alpha_power': alpha_power,
                'beta_power': beta_power,
                'gamma_power': gamma_power
            }
            participant_data.append(combined_features)

        output_df = pd.DataFrame(participant_data)
        save_path = os.path.join(channel_dir, f's{participant:02d}_bandpower.csv')
        output_df.to_csv(save_path, index=False)

        print(f"Band power features for participant s{participant:02d} in channel {channel_name} saved at {save_path}")

# **Classification**



# ANN

1. Valence

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- SOLUTION PART 1: Define separate paths ---
# Path for the metadata file (participant ratings)
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
# Path for the processed channel data created by your first script
channel_data_folder_path = '/kaggle/working/'

# Read targets from the correct metadata folder
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

class EmotionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.relu = nn.LeakyReLU()
        self.bn1 = nn.BatchNorm1d(64)
        self.lastlayer = nn.Linear(32, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.bn1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.lastlayer(x)
        return torch.sigmoid(x)

def crossval_train(x_train, y_train, num_epochs, learning_rate, batch_size, num_classes, n_folds, verbose=False):
    skf = StratifiedKFold(n_splits=n_folds)
    best_models = []
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    for train_index, val_index in skf.split(x_train.cpu().numpy(), y_train.cpu().numpy()):
        x_tr, x_val = x_train[train_index], x_train[val_index]
        y_tr, y_val = y_train[train_index], y_train[val_index]

        # Move data to device
        x_tr, x_val = x_tr.to(device), x_val.to(device)
        y_tr, y_val = y_tr.to(device), y_val.to(device)

        train_set = torch.utils.data.TensorDataset(x_tr, y_tr)
        val_set = torch.utils.data.TensorDataset(x_val, y_val)

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

        model = EmotionClassifier(x_train.shape[1], num_classes).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
        criterion = nn.BCELoss()

        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels)
                loss.backward()
                optimizer.step()

        best_models.append(model)
    return best_models[-1]

def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        predicted = (outputs > 0.5).float()
        accuracy = accuracy_score(y_test.cpu().numpy(), predicted.cpu().numpy())
        precision = precision_score(y_test.cpu().numpy(), predicted.cpu().numpy(), average=None, zero_division=0)
        f1 = f1_score(y_test.cpu().numpy(), predicted.cpu().numpy(), average=None, zero_division=0)
    return accuracy, precision, f1

scaler = StandardScaler()

accuracies = []
precisions_v_minus = []
precisions_v_plus = []
channel_names = []
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(999)

# --- SOLUTION PART 2: Loop over the correct directory ---
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        channel_names.append(ch_name)

        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)

        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            channel_names.pop() # Remove the name we just added
            continue
            
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).to(device)

        best_model = crossval_train(X_train_tensor, y_train_tensor, num_epochs=50, learning_rate=0.001, batch_size=64, num_classes=1, n_folds=5)
        accuracy, precision, f1 = evaluate_model(best_model, X_test_tensor, y_test_tensor)

        accuracies.append(accuracy)
        precisions_v_minus.append(precision[0])
        precisions_v_plus.append(precision[1])

channels = channel_names

# Add a check to ensure we have data before plotting
if not channels:
    print("No channel data was processed. Cannot generate plots.")
else:
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channels, y=accuracies)
    plt.title("Accuracy per Channel")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    precision_df = pd.DataFrame({
        'Channel': channels,
        'Precision V-': precisions_v_minus,
        'Precision V+': precisions_v_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("Precision per Channel (V- and V+)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

# --- FIX 3: Split data ONCE before the loop to get consistent test set ---
# We need to split the data for each channel consistently.
# We'll split the indices first, which is the most robust way.
num_samples = len(y) # Should be 1280 (32 participants * 40 trials)
indices = np.arange(num_samples)

# Split indices into training and testing sets
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# Create the single, consistent y_test for final evaluation
y_test_final = y.iloc[test_indices]


class EmotionClassifier(nn.Module):
    # (Your class definition is correct, no changes needed)
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.relu = nn.LeakyReLU()
        self.bn1 = nn.BatchNorm1d(64)
        self.lastlayer = nn.Linear(32, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.bn1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.lastlayer(x)
        return torch.sigmoid(x)

def crossval_train(x_train, y_train, num_epochs, learning_rate, batch_size, num_classes, n_folds, device, verbose=False):
    # (Your function is mostly correct, just passing device in)
    skf = StratifiedKFold(n_splits=n_folds)
    best_models = []

    for train_index, val_index in skf.split(x_train.cpu().numpy(), y_train.cpu().numpy()):
        x_tr, x_val = x_train[train_index], x_train[val_index]
        y_tr, y_val = y_train[train_index], y_train[val_index]
        
        x_tr, x_val = x_tr.to(device), x_val.to(device)
        y_tr, y_val = y_tr.to(device), y_val.to(device)

        train_set = torch.utils.data.TensorDataset(x_tr, y_tr)
        val_set = torch.utils.data.TensorDataset(x_val, y_val)

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

        model = EmotionClassifier(x_train.shape[1], num_classes).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
        criterion = nn.BCELoss()

        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels)
                loss.backward()
                optimizer.step()
        best_models.append(model)
    return best_models[-1]


def fusion_probabiliste(probabilities_list):
    return np.mean(probabilities_list, axis=0)

def fusion_par_vote(predictions_list):
    predictions_array = np.array(predictions_list)
    return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions_array)

def fusion_bba(probabilities_list):
    # Note: This is a simplified conjunction. For it to be numerically stable,
    # it's better to work in log-space, but this will work for now.
    fused_beliefs = probabilities_list[0]
    for probs in probabilities_list[1:]:
        # Add a small epsilon to avoid division by zero
        denominator = (fused_beliefs * probs + (1 - fused_beliefs) * (1 - probs))
        denominator[denominator == 0] = 1e-9 
        fused_beliefs = fused_beliefs * probs / denominator
    return fused_beliefs


# --- Main Processing Loop ---
scaler = StandardScaler()
probabilities_list = []
predictions_list = []
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(999)

# Loop over the correct data directory
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    # --- FIX 2: Correctly indent the processing logic ---
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files in {ch_name}, skipping.")
            continue

        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        # Drop the trial column if it exists, as it's not a feature
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # y_test is already defined as y_test_final

        # Convert to Tensors
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
        
        # Train a model for this channel
        best_model = crossval_train(X_train_tensor, y_train_tensor, num_epochs=50, learning_rate=0.001, batch_size=64, num_classes=1, n_folds=5, device=device)

        # Get predictions ON THE CONSISTENT TEST SET
        best_model.eval()
        with torch.no_grad():
            probabilities = best_model(X_test_tensor).cpu().numpy().squeeze()
            predictions = (probabilities > 0.5).astype(int)

        probabilities_list.append(probabilities)
        predictions_list.append(predictions)

# --- Final Evaluation (Now outside the loop) ---
# Check if any models were trained before proceeding
if not probabilities_list:
    print("ERROR: No channel data was processed. Cannot perform fusion.")
else:
    fused_probabilities = fusion_probabiliste(probabilities_list)
    fused_predictions_prob = (fused_probabilities > 0.5).astype(int)
    # Use the consistent y_test_final
    accuracy_prob = accuracy_score(y_test_final, fused_predictions_prob)
    print(f"Accuracy after probabilistic fusion: {accuracy_prob:.2f}")

    fused_predictions_vote = fusion_par_vote(predictions_list)
    accuracy_vote = accuracy_score(y_test_final, fused_predictions_vote)
    print(f"Accuracy after majority vote fusion: {accuracy_vote:.2f}")

    fused_beliefs = fusion_bba(probabilities_list)
    fused_predictions_bba = (fused_beliefs > 0.5).astype(int)
    accuracy_bba = accuracy_score(y_test_final, fused_predictions_bba)
    print(f"Accuracy after belief-based fusion: {accuracy_bba:.2f}")

    fusion_results = pd.DataFrame({
        'Fusion Method': ['Probabilistic', 'Vote', 'Belief'],
        'Accuracy': [accuracy_prob, accuracy_vote, accuracy_bba]
    })

    sns.barplot(data=fusion_results, x='Fusion Method', y='Accuracy')
    plt.title('Comparison of Fusion Methods')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0) # Set y-axis to be between 0 and 1 for accuracy
    plt.show()

2. Arousal

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


class EmotionClassifier(nn.Module):
    # (Your class definition is correct)
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.relu = nn.LeakyReLU()
        self.bn1 = nn.BatchNorm1d(64)
        self.lastlayer = nn.Linear(32, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.bn1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.lastlayer(x)
        return torch.sigmoid(x)

# --- FIX 3: Pass 'device' as an argument for cleaner code ---
def crossval_train(x_train, y_train, num_epochs, learning_rate, batch_size, num_classes, n_folds, device, verbose=False):
    skf = StratifiedKFold(n_splits=n_folds)
    best_models = []

    for train_index, val_index in skf.split(x_train.cpu().numpy(), y_train.cpu().numpy()):
        x_tr, x_val = x_train[train_index], x_train[val_index]
        y_tr, y_val = y_train[train_index], y_train[val_index]
        
        # Move data to the specified device
        x_tr, x_val = x_tr.to(device), x_val.to(device)
        y_tr, y_val = y_tr.to(device), y_val.to(device)

        train_set = torch.utils.data.TensorDataset(x_tr, y_tr)
        val_set = torch.utils.data.TensorDataset(x_val, y_val)
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

        model = EmotionClassifier(x_train.shape[1], num_classes).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
        criterion = nn.BCELoss()

        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels)
                loss.backward()
                optimizer.step()
        best_models.append(model)
    return best_models[-1]

def evaluate_model(model, X_test_tensor, y_test_tensor):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        predicted = (outputs > 0.5).float()
        
        # Move tensors to CPU for scikit-learn functions
        y_true_cpu = y_test_tensor.cpu().numpy()
        y_pred_cpu = predicted.cpu().numpy()

        accuracy = accuracy_score(y_true_cpu, y_pred_cpu)
        precision = precision_score(y_true_cpu, y_pred_cpu, average=None, zero_division=0)
        f1 = f1_score(y_true_cpu, y_pred_cpu, average=None, zero_division=0)

    return accuracy, precision, f1

# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_a_minus = []
precisions_a_plus = []
channel_names = []

# Define device once, before the loop
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(999)
print(f"Using device: {device}")

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue

        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 4: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # y_test is our y_test_final defined outside the loop

        # Convert to Tensors
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test_final.values, dtype=torch.float32).to(device)

        # Train a model for this channel
        best_model = crossval_train(X_train_tensor, y_train_tensor, num_epochs=50, learning_rate=0.001, batch_size=64, num_classes=1, n_folds=5, device=device)
        
        # Evaluate the model on the consistent test set
        accuracy, precision, f1 = evaluate_model(best_model, X_test_tensor, y_test_tensor)

        accuracies.append(accuracy)
        precisions_a_minus.append(precision[0])
        precisions_a_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    channels = channel_names

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channels, y=accuracies)
    plt.title("Accuracy per Channel (Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    precision_df = pd.DataFrame({
        'Channel': channels,
        'Precision A-': precisions_a_minus,
        'Precision A+': precisions_a_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("Precision per Channel (A- and A+ for Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets and create the y vector for Arousal
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for final evaluation
y_test_final = y.iloc[test_indices]


class EmotionClassifier(nn.Module):
    # (Your class definition is correct)
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.relu = nn.LeakyReLU()
        self.bn1 = nn.BatchNorm1d(64)
        self.lastlayer = nn.Linear(32, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.bn1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.lastlayer(x)
        return torch.sigmoid(x)

# --- FIX 3: Pass 'device' as an argument for cleaner code ---
def crossval_train(x_train, y_train, num_epochs, learning_rate, batch_size, num_classes, n_folds, device, verbose=False):
    skf = StratifiedKFold(n_splits=n_folds)
    best_models = []

    for train_index, val_index in skf.split(x_train.cpu().numpy(), y_train.cpu().numpy()):
        x_tr, x_val = x_train[train_index], x_train[val_index]
        y_tr, y_val = y_train[train_index], y_train[val_index]
        
        x_tr, x_val = x_tr.to(device), x_val.to(device)
        y_tr, y_val = y_tr.to(device), y_val.to(device)

        train_set = torch.utils.data.TensorDataset(x_tr, y_tr)
        val_set = torch.utils.data.TensorDataset(x_val, y_val)
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

        model = EmotionClassifier(x_train.shape[1], num_classes).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
        criterion = nn.BCELoss()

        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), labels)
                loss.backward()
                optimizer.step()
        best_models.append(model)
    return best_models[-1]


# --- Fusion Functions ---
def fusion_probabiliste(probabilities_list):
    return np.mean(probabilities_list, axis=0)

def fusion_par_vote(predictions_list):
    predictions_array = np.array(predictions_list)
    return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions_array)

def fusion_bba(probabilities_list):
    fused_beliefs = probabilities_list[0]
    for probs in probabilities_list[1:]:
        # Add a small epsilon to avoid division by zero
        denominator = (fused_beliefs * probs + (1 - fused_beliefs) * (1 - probs))
        denominator[denominator == 0] = 1e-9 
        fused_beliefs = fused_beliefs * probs / denominator
    return fused_beliefs

# --- Main Processing Loop ---
scaler = StandardScaler()
probabilities_list = []
predictions_list = []
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(999)
print(f"Using device: {device}")

# Loop over the correct data directory
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    # --- FIX 4: Correctly indent the processing logic ---
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files in {ch_name}, skipping.")
            continue

        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        # --- FIX 5: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # y_test is our y_test_final defined outside the loop

        # Convert to Tensors
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
        
        # Train a model for this channel
        best_model = crossval_train(X_train_tensor, y_train_tensor, num_epochs=50, learning_rate=0.001, batch_size=64, num_classes=1, n_folds=5, device=device)

        # Get predictions ON THE CONSISTENT TEST SET
        best_model.eval()
        with torch.no_grad():
            probabilities = best_model(X_test_tensor).cpu().numpy().squeeze()
            predictions = (probabilities > 0.5).astype(int)

        probabilities_list.append(probabilities)
        predictions_list.append(predictions)

# --- Final Evaluation (Now outside the loop) ---
# Check if any models were trained before proceeding
if not probabilities_list:
    print("\nERROR: No channel data was processed. Cannot perform fusion.")
else:
    fused_probabilities = fusion_probabiliste(probabilities_list)
    fused_predictions_prob = (fused_probabilities > 0.5).astype(int)
    # Use the consistent y_test_final
    accuracy_prob = accuracy_score(y_test_final, fused_predictions_prob)
    print(f"\nAccuracy after probabilistic fusion (Arousal): {accuracy_prob:.2f}")

    fused_predictions_vote = fusion_par_vote(predictions_list)
    accuracy_vote = accuracy_score(y_test_final, fused_predictions_vote)
    print(f"Accuracy after majority vote fusion (Arousal): {accuracy_vote:.2f}")

    fused_beliefs = fusion_bba(probabilities_list)
    fused_predictions_bba = (fused_beliefs > 0.5).astype(int)
    accuracy_bba = accuracy_score(y_test_final, fused_predictions_bba)
    print(f"Accuracy after belief-based fusion (Arousal): {accuracy_bba:.2f}")

    fusion_results = pd.DataFrame({
        'Fusion Method': ['Probabilistic', 'Vote', 'Belief'],
        'Accuracy': [accuracy_prob, accuracy_vote, accuracy_bba]
    })

    sns.barplot(data=fusion_results, x='Fusion Method', y='Accuracy')
    plt.title('Comparison of Fusion Methods for Arousal')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0) # Set y-axis to be between 0 and 1 for accuracy
    plt.show()

# SVM

1. Valence

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


def evaluate_svm(model, X_test, y_test):
    predicted = model.predict(X_test)
    accuracy = accuracy_score(y_test, predicted)
    precision = precision_score(y_test, predicted, average=None, zero_division=0)
    f1 = f1_score(y_test, predicted, average=None, zero_division=0)
    return accuracy, precision, f1

# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_v_minus = []
precisions_v_plus = []
channel_names = []

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 3: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- SVM Training with Grid Search ---
        param_grid = {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 0.001, 0.01, 0.1, 1],
            'kernel': ['rbf'] # It's often better to test one kernel at a time
        }
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid = GridSearchCV(SVC(probability=True), param_grid, cv=StratifiedKFold(n_splits=5), scoring='accuracy', n_jobs=-1, refit=True)
        grid.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid.best_params_}")
        best_model = grid.best_estimator_
        
        # Evaluate the model on the consistent test set
        accuracy, precision, f1 = evaluate_svm(best_model, X_test, y_test_final)

        accuracies.append(accuracy)
        precisions_v_minus.append(precision[0])
        precisions_v_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channel_names, y=accuracies)
    plt.title("SVM Accuracy per Channel (Valence)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    precision_df = pd.DataFrame({
        'Channel': channel_names,
        'Precision V-': precisions_v_minus,
        'Precision V+': precisions_v_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("SVM Precision per Channel (V- and V+ for Valence)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

2. Arousal

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets and create the y vector for Arousal
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


def evaluate_svm(model, X_test, y_test):
    predicted = model.predict(X_test)
    accuracy = accuracy_score(y_test, predicted)
    precision = precision_score(y_test, predicted, average=None, zero_division=0)
    f1 = f1_score(y_test, predicted, average=None, zero_division=0)
    return accuracy, precision, f1

# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_a_minus = []
precisions_a_plus = []
channel_names = []

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 3: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- SVM Training with Grid Search ---
        param_grid = {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 0.001, 0.01, 0.1], # Reduced gamma range for faster search
            'kernel': ['rbf'] 
        }
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid = GridSearchCV(SVC(probability=True), param_grid, cv=StratifiedKFold(n_splits=5), scoring='accuracy', n_jobs=-1, refit=True)
        grid.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid.best_params_}")
        best_model = grid.best_estimator_
        
        # Evaluate the model on the consistent test set
        accuracy, precision, f1 = evaluate_svm(best_model, X_test, y_test_final)

        accuracies.append(accuracy)
        precisions_a_minus.append(precision[0])
        precisions_a_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channel_names, y=accuracies)
    plt.title("SVM Accuracy per Channel (Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    # --- FIX 4: Correct the variable names for the DataFrame ---
    precision_df = pd.DataFrame({
        'Channel': channel_names,
        'Precision A-': precisions_a_minus,
        'Precision A+': precisions_a_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("SVM Precision per Channel (A- and A+ for Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

# Random Forest

1. Valence

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_v_minus = []
precisions_v_plus = []
channel_names = []

# Define the hyperparameter grid for the Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 3: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- Random Forest Training with Grid Search ---
        rf = RandomForestClassifier(random_state=42)
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_rf = grid_search.best_estimator_

        # Evaluate the best model on the consistent test set
        y_pred = best_rf.predict(X_test)
        accuracy = accuracy_score(y_test_final, y_pred)
        precision = precision_score(y_test_final, y_pred, average=None, zero_division=0)

        accuracies.append(accuracy)
        precisions_v_minus.append(precision[0])
        precisions_v_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    channels = channel_names

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channels, y=accuracies)
    plt.title("Random Forest Accuracy per Channel (Valence)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    precision_df = pd.DataFrame({
        'Channel': channels,
        'Precision V-': precisions_v_minus,
        'Precision V+': precisions_v_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("Random Forest Precision per Channel (V- and V+ for Valence)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# --- Fusion Functions (with a fix for BBA) ---
def fusion_probabiliste(probabilities_list):
    return np.mean(probabilities_list, axis=0)

def fusion_par_vote(predictions_list):
    predictions_array = np.array(predictions_list)
    return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions_array)

def fusion_bba(probabilities_list):
    # This is a simplified conjunction. Adding an epsilon makes it more stable.
    fused_beliefs = probabilities_list[0]
    for probs in probabilities_list[1:]:
        denominator = (fused_beliefs * probs + (1 - fused_beliefs) * (1 - probs))
        # FIX 5: Avoid division by zero
        denominator[denominator == 0] = 1e-9 
        fused_beliefs = fused_beliefs * probs / denominator
    return fused_beliefs

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for final evaluation
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
probabilities_list = []
predictions_list = []
channel_names = []

# Define the hyperparameter grid for the Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 4: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- Random Forest Training with Grid Search ---
        rf = RandomForestClassifier(random_state=42)
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_rf = grid_search.best_estimator_

        # Get probabilities and predictions on the consistent test set
        probabilities = best_rf.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        predictions = best_rf.predict(X_test)
        
        probabilities_list.append(probabilities)
        predictions_list.append(predictions)

# --- Final Evaluation (Now outside the loop) ---
# Check if any models were trained before proceeding
if not probabilities_list:
    print("\nERROR: No channel data was processed. Cannot perform fusion.")
else:
    fused_probabilities = fusion_probabiliste(probabilities_list)
    fused_predictions_prob = (fused_probabilities > 0.5).astype(int)
    # --- FIX 3: Use the consistent y_test_final for evaluation ---
    accuracy_prob = accuracy_score(y_test_final, fused_predictions_prob)
    print(f"\nAccuracy after probabilistic fusion (Valence): {accuracy_prob:.2f}")

    fused_predictions_vote = fusion_par_vote(predictions_list)
    accuracy_vote = accuracy_score(y_test_final, fused_predictions_vote)
    print(f"Accuracy after majority vote fusion (Valence): {accuracy_vote:.2f}")

    fused_beliefs = fusion_bba(probabilities_list)
    fused_predictions_bba = (fused_beliefs > 0.5).astype(int)
    accuracy_bba = accuracy_score(y_test_final, fused_predictions_bba)
    print(f"Accuracy after belief-based fusion (Valence): {accuracy_bba:.2f}")

    fusion_results = pd.DataFrame({
        'Fusion Method': ['Probabilistic', 'Vote', 'Belief'],
        'Accuracy': [accuracy_prob, accuracy_vote, accuracy_bba]
    })

    sns.barplot(data=fusion_results, x='Fusion Method', y='Accuracy')
    plt.title('Comparison of Fusion Methods for Valence (Random Forest)')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.show()

2. Arousal

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder for Arousal
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_a_minus = []
precisions_a_plus = []
channel_names = []

# Define the hyperparameter grid for the Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 3: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- Random Forest Training with Grid Search ---
        rf = RandomForestClassifier(random_state=42)
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_rf = grid_search.best_estimator_

        # Evaluate the best model on the consistent test set
        y_pred = best_rf.predict(X_test)
        accuracy = accuracy_score(y_test_final, y_pred)
        precision = precision_score(y_test_final, y_pred, average=None, zero_division=0)

        accuracies.append(accuracy)
        precisions_a_minus.append(precision[0])
        precisions_a_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    channels = channel_names

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channels, y=accuracies)
    plt.title("Random Forest Accuracy per Channel (Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    # --- FIX 4: Correct the variable names for the DataFrame ---
    precision_df = pd.DataFrame({
        'Channel': channels,
        'Precision A-': precisions_a_minus,
        'Precision A+': precisions_a_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("Random Forest Precision per Channel (A- and A+ for Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- Fusion Functions (with a fix for BBA) ---
def fusion_probabiliste(probabilities_list):
    return np.mean(probabilities_list, axis=0)

def fusion_par_vote(predictions_list):
    predictions_array = np.array(predictions_list)
    return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions_array)

def fusion_bba(probabilities_list):
    # This is a simplified conjunction. Adding an epsilon makes it more stable.
    fused_beliefs = probabilities_list[0]
    for probs in probabilities_list[1:]:
        denominator = (fused_beliefs * probs + (1 - fused_beliefs) * (1 - probs))
        # FIX 5: Avoid division by zero
        denominator[denominator == 0] = 1e-9 
        fused_beliefs = fused_beliefs * probs / denominator
    return fused_beliefs

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder for Arousal
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for final evaluation
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
probabilities_list = []
predictions_list = []
channel_names = []

# Define the hyperparameter grid for the Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 4: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- Random Forest Training with Grid Search ---
        rf = RandomForestClassifier(random_state=42)
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_rf = grid_search.best_estimator_

        # Get probabilities and predictions on the consistent test set
        probabilities = best_rf.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        predictions = best_rf.predict(X_test)
        
        probabilities_list.append(probabilities)
        predictions_list.append(predictions)

# --- Final Evaluation (Now outside the loop) ---
# Check if any models were trained before proceeding
if not probabilities_list:
    print("\nERROR: No channel data was processed. Cannot perform fusion.")
else:
    fused_probabilities = fusion_probabiliste(probabilities_list)
    fused_predictions_prob = (fused_probabilities > 0.5).astype(int)
    # --- FIX 3: Use the consistent y_test_final for evaluation ---
    accuracy_prob = accuracy_score(y_test_final, fused_predictions_prob)
    print(f"\nAccuracy after probabilistic fusion (Arousal): {accuracy_prob:.2f}")

    fused_predictions_vote = fusion_par_vote(predictions_list)
    accuracy_vote = accuracy_score(y_test_final, fused_predictions_vote)
    print(f"Accuracy after majority vote fusion (Arousal): {accuracy_vote:.2f}")

    fused_beliefs = fusion_bba(probabilities_list)
    fused_predictions_bba = (fused_beliefs > 0.5).astype(int)
    accuracy_bba = accuracy_score(y_test_final, fused_predictions_bba)
    print(f"Accuracy after belief-based fusion (Arousal): {accuracy_bba:.2f}")

    fusion_results = pd.DataFrame({
        'Fusion Method': ['Probabilistic', 'Vote', 'Belief'],
        'Accuracy': [accuracy_prob, accuracy_vote, accuracy_bba]
    })

    sns.barplot(data=fusion_results, x='Fusion Method', y='Accuracy')
    plt.title('Comparison of Fusion Methods for Arousal (Random Forest)')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.show()

# KNN

1. Valence

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder for Valence
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_v_minus = []
precisions_v_plus = []
channel_names = []

# Define the hyperparameter grid for the KNN classifier
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 3: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- KNN Training with Grid Search ---
        knn = KNeighborsClassifier()
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_knn = grid_search.best_estimator_

        # Evaluate the best model on the consistent test set
        y_pred = best_knn.predict(X_test)
        accuracy = accuracy_score(y_test_final, y_pred)
        precision = precision_score(y_test_final, y_pred, average=None, zero_division=0)

        accuracies.append(accuracy)
        precisions_v_minus.append(precision[0])
        precisions_v_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    channels = channel_names

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channels, y=accuracies)
    plt.title("KNN Accuracy per Channel (Valence)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    precision_df = pd.DataFrame({
        'Channel': channels,
        'Precision V-': precisions_v_minus,
        'Precision V+': precisions_v_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("KNN Precision per Channel (V- and V+ for Valence)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# --- Fusion Functions (with a fix for BBA) ---
def fusion_probabiliste(probabilities_list):
    return np.mean(probabilities_list, axis=0)

def fusion_par_vote(predictions_list):
    predictions_array = np.array(predictions_list)
    return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions_array)

def fusion_bba(probabilities_list):
    # This is a simplified conjunction. Adding an epsilon makes it more stable.
    fused_beliefs = probabilities_list[0]
    for probs in probabilities_list[1:]:
        denominator = (fused_beliefs * probs + (1 - fused_beliefs) * (1 - probs))
        # FIX 5: Avoid division by zero
        denominator[denominator == 0] = 1e-9 
        fused_beliefs = fused_beliefs * probs / denominator
    return fused_beliefs

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder for Valence
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
valence_targets = targets['Valence']
y = (valence_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for final evaluation
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
probabilities_list = []
predictions_list = []
channel_names = []

# Define the hyperparameter grid for the KNN classifier
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'] # p=2 is euclidean, p=1 is manhattan
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 4: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- KNN Training with Grid Search ---
        knn = KNeighborsClassifier()
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_knn = grid_search.best_estimator_

        # Get probabilities and predictions on the consistent test set
        probabilities = best_knn.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        predictions = best_knn.predict(X_test)
        
        probabilities_list.append(probabilities)
        predictions_list.append(predictions)

# --- Final Evaluation (Now outside the loop) ---
# Check if any models were trained before proceeding
if not probabilities_list:
    print("\nERROR: No channel data was processed. Cannot perform fusion.")
else:
    fused_probabilities = fusion_probabiliste(probabilities_list)
    fused_predictions_prob = (fused_probabilities > 0.5).astype(int)
    # --- FIX 3: Use the consistent y_test_final for evaluation ---
    accuracy_prob = accuracy_score(y_test_final, fused_predictions_prob)
    print(f"\nAccuracy after probabilistic fusion (Valence): {accuracy_prob:.2f}")

    fused_predictions_vote = fusion_par_vote(predictions_list)
    accuracy_vote = accuracy_score(y_test_final, fused_predictions_vote)
    print(f"Accuracy after majority vote fusion (Valence): {accuracy_vote:.2f}")

    fused_beliefs = fusion_bba(probabilities_list)
    fused_predictions_bba = (fused_beliefs > 0.5).astype(int)
    accuracy_bba = accuracy_score(y_test_final, fused_predictions_bba)
    print(f"Accuracy after belief-based fusion (Valence): {accuracy_bba:.2f}")

    fusion_results = pd.DataFrame({
        'Fusion Method': ['Probabilistic', 'Vote', 'Belief-Based'],
        'Accuracy': [accuracy_prob, accuracy_vote, accuracy_bba]
    })

    sns.barplot(data=fusion_results, x='Fusion Method', y='Accuracy')
    plt.title('Comparison of Fusion Methods for Valence (KNN)')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.show()

2. Arousal

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score
import seaborn as sns
import matplotlib.pyplot as plt

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder for Arousal
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for testing all models
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
accuracies = []
precisions_a_minus = []
precisions_a_plus = []
channel_names = []

# Define the hyperparameter grid for the KNN classifier
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'] # p=2 is euclidean, p=1 is manhattan
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 3: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- KNN Training with Grid Search ---
        knn = KNeighborsClassifier()
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_knn = grid_search.best_estimator_

        # Evaluate the best model on the consistent test set
        y_pred = best_knn.predict(X_test)
        accuracy = accuracy_score(y_test_final, y_pred)
        precision = precision_score(y_test_final, y_pred, average=None, zero_division=0)

        accuracies.append(accuracy)
        precisions_a_minus.append(precision[0])
        precisions_a_plus.append(precision[1])

# --- Plotting Results ---
# Check if any data was processed before trying to plot
if not channel_names:
    print("\nERROR: No channel data was processed. Cannot generate plots.")
else:
    channels = channel_names

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=channels, y=accuracies)
    plt.title("KNN Accuracy per Channel (Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for accuracy

    for i, value in enumerate(accuracies):
        ax.text(i, value + 0.01, f'{value:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    precision_df = pd.DataFrame({
        'Channel': channels,
        'Precision A-': precisions_a_minus,
        'Precision A+': precisions_a_plus
    })

    precision_df.set_index('Channel', inplace=True)
    precision_df.plot(kind='bar', figsize=(12, 6), color=['blue', 'orange'])

    plt.title("KNN Precision per Channel (A- and A+ for Arousal)")
    plt.xlabel("Channel")
    plt.ylabel("Precision")
    plt.xticks(rotation=90)
    plt.ylim(0, 1.0) # Set a consistent y-axis for precision
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# --- Fusion Functions (with a fix for BBA) ---
def fusion_probabiliste(probabilities_list):
    return np.mean(probabilities_list, axis=0)

def fusion_par_vote(predictions_list):
    predictions_array = np.array(predictions_list)
    return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions_array)

def fusion_bba(probabilities_list):
    # This is a simplified conjunction. Adding an epsilon makes it more stable.
    fused_beliefs = probabilities_list[0]
    for probs in probabilities_list[1:]:
        denominator = (fused_beliefs * probs + (1 - fused_beliefs) * (1 - probs))
        # FIX 5: Avoid division by zero
        denominator[denominator == 0] = 1e-9 
        fused_beliefs = fused_beliefs * probs / denominator
    return fused_beliefs

# --- FIX 1: Define correct paths for metadata and channel data ---
metadata_folder_path = '/kaggle/input/deap-dataset/deap-dataset/Metadata/'
channel_data_folder_path = '/kaggle/working/' # Your processed data is here

# Read targets from the metadata folder for Arousal
targets = pd.read_excel(os.path.join(metadata_folder_path, 'participant_ratings.xls'), index_col=0)
arousal_targets = targets['Arousal']
y = (arousal_targets >= 4.5).astype(int)

# --- FIX 2: Split data ONCE before the loop for a consistent test set ---
# We split indices, which is a robust way to ensure all channels use the same split.
num_samples = len(y) # Total samples = 32 participants * 40 trials = 1280
indices = np.arange(num_samples)

# Split indices into training and testing sets, stratifying on the labels
train_indices, test_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)

# This is our single, consistent set of true labels for final evaluation
y_test_final = y.iloc[test_indices]


# --- Main Processing Loop ---
scaler = StandardScaler()
probabilities_list = []
predictions_list = []
channel_names = []

# Define the hyperparameter grid for the KNN classifier
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'] # p=2 is euclidean, p=1 is manhattan
}

# Loop over the correct directory containing channel data
for ch_name in os.listdir(channel_data_folder_path):
    channel_full_path = os.path.join(channel_data_folder_path, ch_name)
    
    if os.path.isdir(channel_full_path):
        print(f"Processing channel: {ch_name}")
        ch_data = []
        for file_name in os.listdir(channel_full_path):
            if file_name.endswith('.csv'):
                file_path = os.path.join(channel_full_path, file_name)
                file_data = pd.read_csv(file_path)
                ch_data.append(file_data)
        
        if not ch_data:
            print(f"  - No CSV files found in {ch_name}, skipping.")
            continue
            
        channel_names.append(ch_name)
        ch_data = pd.concat(ch_data, axis=0, ignore_index=True)
        
        # --- FIX 4: Remove non-feature columns before creating X ---
        if 'trial' in ch_data.columns:
            ch_data = ch_data.drop(columns=['trial'])
            
        X = ch_data.values
        X_scaled = scaler.fit_transform(X)

        # Use the pre-split indices to get consistent data subsets
        X_train = X_scaled[train_indices]
        X_test = X_scaled[test_indices]
        y_train = y.iloc[train_indices]
        # We will use y_test_final for evaluation

        # --- KNN Training with Grid Search ---
        knn = KNeighborsClassifier()
        # Using n_jobs=-1 will use all available CPU cores, speeding up the search
        grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, scoring='accuracy', cv=StratifiedKFold(n_splits=5), verbose=0, n_jobs=-1, refit=True)
        grid_search.fit(X_train, y_train)

        print(f"  - Best params for {ch_name}: {grid_search.best_params_}")
        best_knn = grid_search.best_estimator_

        # Get probabilities and predictions on the consistent test set
        probabilities = best_knn.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        predictions = best_knn.predict(X_test)
        
        probabilities_list.append(probabilities)
        predictions_list.append(predictions)

# --- Final Evaluation (Now outside the loop) ---
# Check if any models were trained before proceeding
if not probabilities_list:
    print("\nERROR: No channel data was processed. Cannot perform fusion.")
else:
    fused_probabilities = fusion_probabiliste(probabilities_list)
    fused_predictions_prob = (fused_probabilities > 0.5).astype(int)
    # --- FIX 3: Use the consistent y_test_final for evaluation ---
    accuracy_prob = accuracy_score(y_test_final, fused_predictions_prob)
    print(f"\nAccuracy after probabilistic fusion (Arousal): {accuracy_prob:.2f}")

    fused_predictions_vote = fusion_par_vote(predictions_list)
    accuracy_vote = accuracy_score(y_test_final, fused_predictions_vote)
    print(f"Accuracy after majority vote fusion (Arousal): {accuracy_vote:.2f}")

    fused_beliefs = fusion_bba(probabilities_list)
    fused_predictions_bba = (fused_beliefs > 0.5).astype(int)
    accuracy_bba = accuracy_score(y_test_final, fused_predictions_bba)
    print(f"Accuracy after belief-based fusion (Arousal): {accuracy_bba:.2f}")

    fusion_results = pd.DataFrame({
        'Fusion Method': ['Probabilistic', 'Vote', 'Belief-Based'],
        'Accuracy': [accuracy_prob, accuracy_vote, accuracy_bba]
    })

    sns.barplot(data=fusion_results, x='Fusion Method', y='Accuracy')
    plt.title('Comparison of Fusion Methods for Arousal (KNN)')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.0)
    plt.show()

# Fusion au niveau des caractéristiques (Features Fusion)


# **Feature Extraction and labeling**

In [ ]:
eeg_data = data[:,:32,:]
print(eeg_data.shape)

In [ ]:
import numpy as np
from scipy.signal import welch
# FIX 1: Import 'trapezoid' instead of the old 'simps'
from scipy.integrate import trapezoid

def bandpower(data, sf, band):
    band = np.asarray(band)
    low, high = band
    # Ensure nperseg is at least 2, handle potential division by zero if low is 0
    if low == 0:
        low = 1e-6 # Avoid division by zero, use a very small number
    nperseg = (2 / low) * sf
    
    freqs, psd = welch(data, sf, nperseg=nperseg)
    freq_res = freqs[1] - freqs[0]
    idx_band = np.logical_and(freqs >= low, freqs <= high)
    
    # FIX 2: Call 'trapezoid' instead of 'simps'
    bp = trapezoid(psd[idx_band], dx=freq_res)
    return bp

# This function relies on a global variable 'eeg_data'.
# Make sure 'eeg_data' is defined before calling this function.
def get_band_power(people, channel, band):
    bd = (0,0)
    if (band == "alpha"):
        bd = (8,12)
    elif (band == "beta"):
        bd = (12,30)
    elif (band == "gamma"):
        bd = (30,64)
    
    # This will raise a NameError if 'eeg_data' is not defined in the global scope.
    return bandpower(eeg_data[people,channel], 128, bd)

In [ ]:
eeg_band = []
for i in range (len(eeg_data)):
    for j in range (len(eeg_data[0])):
        eeg_band.append(get_band_power(i,j,"alpha"))
        eeg_band.append(get_band_power(i,j,"beta"))
        eeg_band.append(get_band_power(i,j,"gamma"))

print(eeg_band)

In [ ]:
eeg_band = np.array(eeg_band)
eeg_band = eeg_band.reshape((1280,3)) # 5×32
print(eeg_band.shape)
print(eeg_band)

In [ ]:
np.save("eeg_band.npy", eeg_band)
eeg_band = np.load("eeg_band.npy")
print(eeg_band.shape)

## Construire les données de l’étiquette

In [ ]:
import pandas as pd

df_label = pd.DataFrame({'Valence': labels[:,0], 'Arousal': labels[:,1],
                        'Dominance': labels[:,2], 'Liking': labels[:,3]})
df_label

In [ ]:
df_label.info()

In [ ]:
df_label.describe()

In [ ]:
label_name = ["valence","arousal"]
labels_valence = []
labels_arousal = []
for la in labels:
    l = []
    if la[0]>4.5:
        labels_valence.append(1)
    else:
        labels_valence.append(0)
    if la[1]>4.5:
        labels_arousal.append(1)
    else:
        labels_arousal.append(0)

# Création de modèles, formation, tests et optimisation

# **Classification**

In [ ]:
#Données X
data_x = eeg_band
print(data_x.shape)

#Données Y
label_y0 = labels_valence
label_y1 = labels_arousal
trainscores = []
testscores = []

## KNN

## Valence

In [ ]:
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

poly = preprocessing.PolynomialFeatures(degree=1)
X = poly.fit_transform(data_x)

scaler = preprocessing.StandardScaler()  # Standardisation
X = scaler.fit_transform(X)

# Réduction de la dimensionnalité
pca = PCA(n_components=0.95)  # Ajuster pour conserver 95% de la variance
X = pca.fit_transform(X)
print(X.shape)

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, label_y0, test_size=0.20, random_state=42)

# Recherche des hyperparamètres
param_grid = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Meilleurs paramètres
print("Meilleurs paramètres :", grid_search.best_params_)

# Modèle optimisé
knn_best = grid_search.best_estimator_
train_score_V_KNN = knn_best.score(X_train, y_train)
test_score_V_KNN = knn_best.score(X_test, y_test)

print("Score de l’ensemble d’entraînement :", train_score_V_KNN)
print("Score de l’ensemble de tests :", test_score_V_KNN)

from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(knn_best, X, label_y0, cv=5, scoring='accuracy')
print("Scores de validation croisée :", cv_scores)
print("Score moyen de validation croisée :", cv_scores.mean())

## Arousal

In [ ]:
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Prétraitement
poly = preprocessing.PolynomialFeatures(degree=1)  # Réduction du degré
X = poly.fit_transform(data_x)

scaler = preprocessing.StandardScaler()  # Standardisation
X = scaler.fit_transform(X)

# Réduction de la dimensionnalité
pca = PCA(n_components=0.95)  # Ajuster pour conserver 95% de la variance
X = pca.fit_transform(X)
print(X.shape)

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, label_y1, test_size=0.20, random_state=42)

# Recherche des hyperparamètres
param_grid = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Meilleurs paramètres
print("Meilleurs paramètres :", grid_search.best_params_)

# Modèle optimisé
knn_best = grid_search.best_estimator_
train_score_A_KNN = knn_best.score(X_train, y_train)
test_score_A_KNN = knn_best.score(X_test, y_test)

print("Score de l’ensemble d’entraînement :", train_score_A_KNN)
print("Score de l’ensemble de tests :", test_score_A_KNN)


from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(knn_best, X, label_y1, cv=5, scoring='accuracy')
print("Scores de validation croisée :", cv_scores)
print("Score moyen de validation croisée :", cv_scores.mean())

In [ ]:
import matplotlib.pyplot as plt

emotions = ['Valence', 'Arousal']
testscores = [test_score_V_KNN, test_score_A_KNN]
plt.figure(figsize=(10,6))
bars = plt.bar(emotions, testscores, color='skyblue')
plt.xlabel('Émotions')
plt.ylabel('Score de Test')
plt.title('Scores de Test pour Différentes Émotions')
plt.ylim(0, 1)

for bar, score in zip(bars, testscores):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{score:.2f}', ha='center', va='bottom', fontsize=12)

plt.show()

## SVM

## Valence

In [ ]:
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
import numpy as np

poly = preprocessing.PolynomialFeatures(degree=1)
X = poly.fit_transform(data_x)

scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(X)

pca = PCA(n_components=0.95)
X = pca.fit_transform(X)
print(X.shape)

X_train, X_test, y_train, y_test = train_test_split(X, label_y0, test_size=0.20, random_state=42)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'linear', 'poly']
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Meilleurs paramètres : {grid_search.best_params_}")

svc = grid_search.best_estimator_
train_score_V_SVM = svc.score(X_train, y_train)
test_score_V_SVM = svc.score(X_test, y_test)

print(f"Score de l’ensemble d’entraînement : {train_score_V_SVM}")
print(f"Score de l’ensemble de tests : {test_score_V_SVM}")

## Arousal

In [ ]:
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
import numpy as np

poly = preprocessing.PolynomialFeatures(degree=1)
X = poly.fit_transform(data_x)

scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(X)

pca = PCA(n_components=0.95)
X = pca.fit_transform(X)
print(X.shape)

X_train, X_test, y_train, y_test = train_test_split(X, label_y1, test_size=0.20, random_state=42)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'linear', 'poly']
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Meilleurs paramètres : {grid_search.best_params_}")

svc = grid_search.best_estimator_
train_score_A_SVM = svc.score(X_train, y_train)
test_score_A_SVM = svc.score(X_test, y_test)

print(f"Score de l’ensemble d’entraînement : {train_score_A_SVM}")
print(f"Score de l’ensemble de tests : {test_score_A_SVM}")

In [ ]:
import matplotlib.pyplot as plt

emotions = ['Valence', 'Arousal']
testscores1 = [test_score_V_SVM, test_score_A_SVM]
plt.figure(figsize=(10,6))
bars = plt.bar(emotions, testscores1, color='skyblue')
plt.xlabel('Émotions')
plt.ylabel('Score de Test')
plt.title('Scores de Test pour Différentes Émotions')
plt.ylim(0, 1)

for bar, score in zip(bars, testscores1):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{score:.2f}', ha='center', va='bottom', fontsize=12)

plt.show()

## Arbre de décision

## Valence

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Prétraitement
scaler = StandardScaler()
X = scaler.fit_transform(data_x)

pca = PCA(n_components=0.95)  # Conserver 95% de la variance
X = pca.fit_transform(X)
print("Nouvelle forme des données :", X.shape)

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, label_y0, test_size=0.20, random_state=42)

# Recherche des hyperparamètres
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 4, 10],
    'min_samples_leaf': [1, 2, 5]
}

dtree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dtree, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Meilleurs hyperparamètres
print("Meilleurs hyperparamètres :", grid_search.best_params_)

# Modèle optimisé
dtree_best = grid_search.best_estimator_
train_score_V_AD = dtree_best.score(X_train, y_train)
test_score_V_AD = dtree_best.score(X_test, y_test)

print("Score de l’ensemble d’entraînement :", train_score_V_AD)
print("Score de l’ensemble de tests :", test_score_V_AD)

# Validation croisée
cv_scores = cross_val_score(dtree_best, X, label_y0, cv=5, scoring='accuracy')
print("Scores de validation croisée :", cv_scores)
print("Score moyen :", cv_scores.mean())

## Arousal

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Prétraitement
scaler = StandardScaler()
X = scaler.fit_transform(data_x)

pca = PCA(n_components=0.95)  # Conserver 95% de la variance
X = pca.fit_transform(X)
print("Nouvelle forme des données :", X.shape)

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, label_y1, test_size=0.20, random_state=42)

# Recherche des hyperparamètres
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 4, 10],
    'min_samples_leaf': [1, 2, 5]
}

dtree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(dtree, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Meilleurs hyperparamètres
print("Meilleurs hyperparamètres :", grid_search.best_params_)

# Modèle optimisé
dtree_best = grid_search.best_estimator_
train_score_A_AD = dtree_best.score(X_train, y_train)
test_score_A_AD = dtree_best.score(X_test, y_test)

print("Score de l’ensemble d’entraînement :", train_score_A_AD)
print("Score de l’ensemble de tests :", test_score_A_AD)

# Validation croisée
cv_scores = cross_val_score(dtree_best, X, label_y1, cv=5, scoring='accuracy')
print("Scores de validation croisée :", cv_scores)
print("Score moyen :", cv_scores.mean())

In [ ]:
import matplotlib.pyplot as plt

emotions = ['Valence', 'Arousal']
testscores1 = [test_score_V_AD, test_score_A_AD]
plt.figure(figsize=(10,6))
bars = plt.bar(emotions, testscores1, color='skyblue')
plt.xlabel('Émotions')
plt.ylabel('Score de Test')
plt.title('Scores de Test pour Différentes Émotions')
plt.ylim(0, 1)

for bar, score in zip(bars, testscores1):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{score:.2f}', ha='center', va='bottom', fontsize=12)

plt.show()

## Random Forest


## Valence

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Assume 'data_x' and 'label_y0' are defined correctly before this
# For example:
# data_x = np.random.rand(1280, 10) 
# label_y0 = np.random.randint(0, 2, 1280)

# 1. Polynomial Feature Generation
poly = preprocessing.PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(data_x)

# 2. Scaling
min_max_scaler = preprocessing.MinMaxScaler()
X_scaled = min_max_scaler.fit_transform(X_poly)

# --- FIX for Data Leakage: Split the data BEFORE applying PCA ---
# We split the preprocessed (scaled) data, not the final reduced data
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(
    X_scaled, label_y0, test_size=0.2, random_state=42, stratify=label_y0
)

# 3. Principal Component Analysis
# --- FIX for ValueError: Use a ratio of variance instead of a fixed number ---
# This is the recommended approach. PCA will choose the number of components
# needed to explain 95% of the variance.
pca = PCA(n_components=0.95)

# Fit PCA ONLY on the training data
pca.fit(X_train_scaled)

# Apply the trained PCA to both train and test sets
X_train_reduced = pca.transform(X_train_scaled)
X_test_reduced = pca.transform(X_test_scaled)

print(f"Original number of features: {X_train_scaled.shape[1]}")
print(f"Number of components chosen by PCA: {pca.n_components_}")


# 4. Model Training and Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1 # Use all available CPU cores
)

# Train on the PCA-reduced training data
grid_search.fit(X_train_reduced, y_train)
best_model = grid_search.best_estimator_

print("\nMeilleurs hyperparamètres :", grid_search.best_params_)

# 5. Evaluation
train_score = best_model.score(X_train_reduced, y_train)
test_score = best_model.score(X_test_reduced, y_test)

print("Score de l’ensemble d’entraînement :", train_score)
print("Score de l’ensemble de test :", test_score)

# For a more robust estimate of performance, cross-validate the entire dataset
# Note: For a truly unbiased estimate, this cross-validation should also
# contain the PCA step inside it using a scikit-learn Pipeline.
# However, this approach is a good approximation.
X_reduced_full = pca.transform(X_scaled) # Apply PCA to the full dataset for this step
cv_scores = cross_val_score(best_model, X_reduced_full, label_y0, cv=5, scoring='accuracy')
print("\nScores de validation croisée :", cv_scores)
print("Score moyen :", cv_scores.mean())

y_pred = best_model.predict(X_test_reduced)
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

## Arousal

In [ ]:
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Assume 'data_x' and 'label_y1' are defined correctly before this.
# For example:
# data_x = np.random.rand(1280, 10) 
# label_y1 = np.random.randint(0, 2, 1280)

# 1. Initial Preprocessing (Polynomial Features and Scaling)
poly = preprocessing.PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(data_x)

min_max_scaler = preprocessing.MinMaxScaler()
X_scaled = min_max_scaler.fit_transform(X_poly)

# --- FIX for Data Leakage: Split the data BEFORE applying PCA ---
# We split the preprocessed (scaled) data.
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(
    X_scaled, label_y1, test_size=0.2, random_state=42, stratify=label_y1
)

# 2. Principal Component Analysis (PCA)
# --- FIX for ValueError: Use a ratio of variance (e.g., 0.95) ---
# This is the recommended approach. PCA will automatically choose the number of
# components needed to explain 95% of the variance in the training data.
# This avoids the error and adapts to your data's characteristics.
pca = PCA(n_components=0.95)

# FIT PCA **ONLY** ON THE TRAINING DATA to prevent data leakage.
pca.fit(X_train_scaled)

# APPLY the trained PCA to both the training and test sets.
X_train_reduced = pca.transform(X_train_scaled)
X_test_reduced = pca.transform(X_test_scaled)

print(f"Original number of features after scaling: {X_train_scaled.shape[1]}")
print(f"Number of components chosen by PCA to preserve 95% variance: {pca.n_components_}")


# 3. Model Training and Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1 # Use all available CPU cores for faster search
)

# Train on the correctly processed and reduced training data.
grid_search.fit(X_train_reduced, y_train)
best_model = grid_search.best_estimator_

print("\nMeilleurs hyperparamètres :", grid_search.best_params_)

# 4. Evaluation
# Evaluate on the correctly transformed train and test sets.
train_score = best_model.score(X_train_reduced, y_train)
test_score = best_model.score(X_test_reduced, y_test)

print("Score de l’ensemble d’entraînement :", train_score)
print("Score de l’ensemble de test :", test_score)

# For cross-validation on the full dataset, we first transform all the data
# using the PCA that was trained ONLY on the original training set.
# This prevents data leakage during this evaluation step as well.
X_reduced_full = pca.transform(X_scaled)
cv_scores = cross_val_score(best_model, X_reduced_full, label_y1, cv=5, scoring='accuracy')
print("\nScores de validation croisée :", cv_scores)
print("Score moyen :", cv_scores.mean())

# Make predictions on the correctly transformed test set.
y_pred = best_model.predict(X_test_reduced)
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

emotions = ['Valence', 'Arousal']
testscores1 = [0.6875, 0.64453125]
plt.figure(figsize=(10,6))
bars = plt.bar(emotions, testscores1, color='skyblue')
plt.xlabel('Émotions')
plt.ylabel('Score de Test')
plt.title('Scores de Test pour Différentes Émotions')
plt.ylim(0, 1)

for bar, score in zip(bars, testscores1):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{score:.2f}', ha='center', va='bottom', fontsize=12)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données
emotions = ['Valence', 'Arousal']
methods = ['KNN', 'SVM', 'Arbre de décision', 'Random Forest']
scores = {
    'KNN': [test_score_V_KNN, test_score_A_KNN],
    'SVM': [test_score_V_SVM, test_score_A_SVM],
    'Arbre de décision': [test_score_V_AD, test_score_A_AD],
    'Random Forest': [0.6875, 0.64453125]
}

# Paramètres du graphique
x = np.arange(len(emotions))
bar_width = 0.2
offsets = np.arange(len(methods)) * bar_width - (len(methods) - 1) * bar_width / 2
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']  # Couleurs des barres

plt.figure(figsize=(12, 7))

# Création des barres
for i, (method, method_scores) in enumerate(scores.items()):
    plt.bar(
        x + offsets[i],
        method_scores,
        bar_width,
        label=method,
        color=colors[i],
        edgecolor='black'  # Bordures noires pour chaque barre
    )

# Configuration des axes
plt.xticks(x, emotions, fontsize=12)
plt.xlabel('Émotions', fontsize=14)
plt.ylabel('Score de Test', fontsize=14)
plt.title('Scores de Test pour Différentes Méthodes et Émotions', fontsize=16)
plt.ylim(0, 1.1)  # Laisser de l'espace au-dessus des barres

# Ajout des annotations
for i, (method, method_scores) in enumerate(scores.items()):
    for j, score in enumerate(method_scores):
        plt.text(
            x[j] + offsets[i],
            score + 0.02,
            f'{score:.2f}',
            ha='center',
            va='bottom',
            fontsize=10,
            color='black'
        )

# Ajout de la légende
plt.legend(title='Méthodes', fontsize=12, title_fontsize=14)

# Optimisation de l'affichage
plt.grid(axis='y', linestyle='--', alpha=0.7)  # Grille horizontale pour faciliter la lecture
plt.tight_layout()

# Affichage
plt.show()